One of the most useful tools in JKAT is the orbital transfer calculator.
Since it is often desirable to get from one orbit to another, this is implemented in several functions with more or less granularity.

First we import what we need:

In [1]:
import jkat
import numpy as np
import matplotlib.pyplot as plt
%matplotlib qt
# set plotting windows to interactive

The basis of orbital transfers is solving the lambert problem. This finds an orbit (or rather the velocities that make up an orbit) connecting two points for a given transfer time.
lambert also needs to know the direction of the orbit (if it's "long" or "short"), by default (`prograde=None`) it always picks the shortest direction, but can also choose the direction counterclockwise around the $+Z$ axis (`prograde=True`), or clockwise around the $+Z$ axis (`prograde=False`).

In [2]:
r1 = np.array([1, 2, 2]) # origin vector
r2 = np.array([-1,-1,-1]) # destination vector
dt = 1000 # transfer time
mu = 500 # standard gravitational parameter
v1, v2 = jkat.trajectories.lambert(r1,r2,dt,mu, prograde=None)

print(v1) # velocity vector at r1
print(v2) # velocity vector at r2

[-4.69244795 12.43321249 12.43321249]
[23.81636946  1.99826108  1.99826108]


Often however, the results will directly be used to calculate an orbit using something like `orbit_from_rv(r1,v1, mu)`, so for that purpose, the `orbit_from_lambert` function does that automatically. Additionally, the `r1` and `r2` vectors often originate from two separate orbits, in which case the `orbit_from_transfer` can be used instead of `r1 = ob1.t2rvec(t0)` etc.

In [3]:
jkat.orbit_from_lambert(r1,r2,0,dt,mu) # quicker way to make orbit (not the slight change in format with start and end times rather than just dt)

Orbit:
 p: 1.9041194127970698
 e: 0.995907823798126
 i: 0.7853981633974484
 raan: 0.0
 argp: 5.5677873647654295
 tp: -0.14845108525226317
 mu: 500

In [4]:
ob1 = jkat.Earth
ob2 = jkat.Mars
t0 = 0
t1 = jkat.YEAR

trans = jkat.orbit_from_transfer(ob1, ob2, t0, t1, prograde=True)

jkat.plot(ob1, color='blue', t=t0)
jkat.plot(ob2, color='red', t=t1)
jkat.plot(trans, t_bounds=(t0,t1), color="green")
jkat.show()

This is all well and good, but as you can see from the plot, the trajectory that was found is very elliptical, and if we calculate the change in velocity $\Delta V$ that would be needed for this transfer we'll see that it's massive:

In [5]:
dv1 = ob1.t2vvec(t0) - trans.t2vvec(t0)
dv2 = ob2.t2vvec(t1) - trans.t2vvec(t1)
total_dv = np.linalg.norm(dv1) + np.linalg.norm(dv2)
print(total_dv, 'km/s')

41.37013049673649 km/s


As we can see, this is massive. Normal $\Delta V$ costs to mars are about 4 km/s for the first burn, and about 2 km/s for the second.
This is way less than the 41 km/s this transfer would take. This is due to the times chosen, `t0` and `t1`, being very unsuitable.
To find correct times without searching manually, the `direct_transfer` function can be used to find an optimum:  

In [6]:
res = jkat.direct_transfer(
    ob1,
    ob2,
    bounds=(0, 2*jkat.YEAR, jkat.YEAR, 2*jkat.YEAR),
    # limit search to a start time between 2000 and 2002 and an arrival
    # time between 2001 and 2002 
    prograde=True # like with lambert, it's possible to choose direction
)
print(res)
trans2 = jkat.orbit_from_transfer(ob1, ob2, res['ts'], res['te'], prograde=True)
jkat.plot(ob1, color='blue', t=res['ts'])
jkat.plot(ob2, color='red', t=res['te'])
jkat.plot(trans2, t_bounds=(res['ts'],res['te']), color="green")
jkat.show()

{'ts': np.float64(35903560.58496451), 'te': np.float64(59373690.011238635), 'tt': np.float64(23470129.426274125), 'dv1': np.float64(3.3546614779626682), 'dv2': np.float64(4.0878962803154995), 'r': np.float64(208139107.0934776)}


This returns a dictionary with an optimised result as a dictionary.
The dictionary includes the start and end times `ts` and `te`, the travel time `tt`, the initial and final $\Delta V$ `dv1` and `dv2`, as well as the altitude of intercept `r`.
By default, direct transfer optimizes for $\Delta V$, summing `dv1` and `dv2` and trying to minimize that number. This behaviour can be changed using several keyword arguments.
All the parameters reported in `res` can be uses as weights, for example the default argument is `dv1_w = 1` and `dv2_w = 1`. to add a limit on travel time, the argument `tt_w = 1/(10*jkat.DAY)` can be passed, this means that every extra 10 days of travel time will be weighted equally to an extra km in $\Delta V$.
Keep in mind that this weight is per unit of whatever parameter is affected, which for default JKAT examples is second, km, and km/s. thus the effects of each weight is not equal.

it is also possible to limit some parameters, for example with `tt_max = jkat.YEAR`, to put a hard limit on this parameter. However bear in mind that putting too many limits on the optimizer may result in it not finding a solution, even where one exists. 
The optimizer will generate some potential optimums with heuristics, additional points can be added with the `suggested_points`, which is a list of start and end time pairs.


To check this result, a thourough search could be done for the $\Delta V$ numbers, plotting this result is what is often called a "prokchop plot", and as such the function to do this is called `porkchop` 

In [ ]:
starttimes= np.linspace(0,2*jkat.YEAR) # this is the same as the bounds in the optimizer
endtimes = np.linspace(jkat.YEAR, 2*jkat.YEAR)

arr = jkat.trajectories.porkchop(
    ob1,
    ob2,
    starttimes,
    endtimes,
    prograde=True
)
idx = np.unravel_index(np.nanargmin(arr), arr.shape) # use numpy to find minimum
opt_point = [starttimes[idx[0]], endtimes[idx[1]]]

plt.imshow(arr.T,
           vmax=30,
           aspect='auto',
           origin='lower',
           extent=(0,2,1,2) # extent in years
)
plt.scatter(res['ts']/jkat.YEAR, res['te']/jkat.YEAR, color='red') # optimal found by direct transfer
plt.scatter(opt_point[0]/jkat.YEAR, opt_point[1]/jkat.YEAR, color = 'blue') # minimum of porkchop
plt.colorbar()
plt.xlabel('start time [years]')
plt.ylabel('end time [years]')
plt.show()
jkat.trajectories.orbit_transfer

(np.int64(31), np.int64(40))


As can be seen, the global search can find other local minimums than the one found by the `direct_transfer` (at the cost of muhc longer compute time).
The inner workings of `direct_transfer` is something that's work-in-progress, and small imporvements are added from time to time.
If the results are unsatisfactory, there should be enough component pieces (e.g. `lambert`) that creating your own optimizer is simple enough.

A final note, if the stransfer is strictly between two _orbits_, and the timing is irrelevant, then `orbit_transfer` may be preferrable as it optimises using true anomaly rather than time.